In [2]:
import pandas as pd
import numpy as np
import time as time_lib
import warnings
warnings.filterwarnings("ignore")

In [3]:
def normal1(df):
  n = df.shape[1]
  maxs = [df[i].max() for i in range(n)]
  df_out = df.copy()
  for i in range(n):
    df_out[i] = (df[i]-1)/(maxs[i]-1)
  return df_out#, maxs

In [4]:
def normal2(df):
  n = df.shape[1]
  maxs = [df[i].max() for i in range(n)]
  df_out = df.copy()
  for i in range(n):
    df_out[i] = df[i]/maxs[i]
  return df_out#, maxs

In [5]:
def unnormal1(df, maxs):
  m = df.shape[0]
  n = df.shape[1]
  df_out = df.copy()
  for i in range(n):
    df_out[i] = df[i]*maxs[i]
  for j in range(n):
    for i in range(m):
      df_out.iloc[i,j] = round(min(maxs[j],max(1, df_out.iloc[i,j])),0)
  return df_out

In [6]:
def normal3(df):
  import numpy as np
  m = df.shape[0]
  n = df.shape[1]
  maxs = [int(df[i].max()) for i in range(n)]
  probs = {}
  for j in range(n):
    mm = m - df.isna().sum(axis=0)[j]
    probs[j] = [np.count_nonzero(df[j] == k)/mm for k in range(1,maxs[j]+1)]
  df_out = df.copy()
  for j in range(n):
    if maxs[j] == 2:
      df_out[j] = df[j]-1
    else:
      c = maxs[j]
      for i in range(m):
        r = df.iloc[i,j]
        if np.isnan(r):
          df_out.iloc[i,j] = np.nan
        elif r == 1:
          df_out.iloc[i,j] = 0
        elif r == 2:
          df_out.iloc[i,j] = (probs[j][0]+probs[j][1])/(sum(probs[j][:c-1])+sum(probs[j][1:c]))
        else:
          df_out.iloc[i,j] = (sum(probs[j][:int(r)-1])+sum(probs[j][1:int(r)]))/(sum(probs[j][:int(c)-1])+sum(probs[j][1:int(c)]))
  return df_out#, maxs, probs

In [7]:
def unnormal3(df,maxs,probs):
  def T2(x, prob, max):
    y = (2*x-prob[0])/(sum(prob[:max-1])+sum(prob[1:]))
    return y
  import numpy as np
  m = df.shape[0]
  n = df.shape[1]
  df_out = df.copy()
  for j in range(n):
    prob = probs[j]
    max = maxs[j]
    if maxs[j] == 2:
      for i in range(m):
        if df.iloc[i,j] < T2(prob[0],prob,max):
          df_out.iloc[i,j] = 1
        else:
          df_out.iloc[i,j] = 2
    else:
      for i in range(m):
        r = df.iloc[i,j]
        if r < T2(prob[0],prob,max):
          df_out.iloc[i,j] = 1
        elif r > T2(sum(prob[:max-1]),prob,max):
          df_out.iloc[i,j] = max
        for k in range(1,max-1):
          if T2(sum(prob[:k]),prob,max) <= r < T2(sum(prob[:k+1]),prob,max):
            df_out.iloc[i,j] = k+1
  return df_out

In [8]:
# A subroutine that calculates $\tau_b$ coefficient for columns $i$ and $j$ of the input dataframe `df`.

def tau_b(df, i, j):
  import scipy.stats as stats
  dat = df.copy()
  keep_ind = []
  for k in range(len(df)):
    if df.isna().iloc[k,i]+df.isna().iloc[k,j] == 0:
      keep_ind.append(k)
  dat_out = dat.iloc[keep_ind,:]
  tau, p = stats.kendalltau(dat_out.iloc[:,i],dat_out.iloc[:,j])
  return tau

In [9]:
def tau_b_mat(df):
  import numpy as np
  n = df.shape[1]
  tau_mat = np.empty(shape=(n, n), dtype='double')
  for i in range(n):
    for j in range(n):
      tau_mat[i,j] = tau_b(df, i, j)
  return tau_mat

# Subroutine: Column-wise aggregation 
It takes a normalized dataframe `df` and the column label `i` and returns the aggregated column by weighted average:
$$
s_i = \frac{\sum_l w_l X_i}{\sum_l w_l}
$$`

In [10]:
def c_agg(df, w):
  import numpy as np
  m = df.shape[0]
  n = df.shape[1]

  agg_mat = np.empty(shape=(m, n), dtype='double')
  masked_df = np.ma.masked_array(df, np.isnan(df))

  for i in range(m):
    for j in range(n):
      agg_mat[i,j] = np.ma.average(masked_df[i,:], weights=w[:,j])
  return agg_mat

Subroutine: Take a column of a missing data and return the imputed one based on monotonization

In [11]:
# new optimization procedure (2023.04.18)
def monotonization(input):

  import numpy as np
  n = len(input)
  c = np.nanmax(input)
  # k = n - np.isnan(input).sum()
  input_obs = input[~np.isnan(input)]
  obs = len(input_obs)
  
  ind = np.where(1-np.isnan(input))
  K = ind[0]

  import gurobipy as gp
  m = gp.Env(empty=True)
  m.setParam('WLSACCESSID', '54ea4edc-45f3-4ac2-b06f-cc4809a3c06c')
  m.setParam('WLSSECRET', '2802dfc5-13c3-4ceb-b6c2-b217b0d5994a')
  m.setParam('LICENSEID', 947226)
  m.setParam("OutputFlag", 0) #add by CHP
  m.start()

# Create the model within the Gurobi environment
  m = gp.Model(env=m)

  from gurobipy import GRB

  y = m.addVars(obs, lb=1, ub=c, name="y")
  u = m.addVars(obs, lb=0, name="u")
  w = m.addVars(obs, lb=0, name="w")

  m.setObjective(gp.quicksum(u[i]+w[i] for i in range(obs)), GRB.MINIMIZE)

  m.addConstr(1 <= y[0])
  m.addConstr(y[obs-1] <= c)
  m.addConstrs(y[i] <= y[i+1] for i in range(obs-1))
  m.addConstrs(y[j]-input_obs[j] == u[j]-w[j] for j in range(obs))

  m.optimize()

  y_out = m.getAttr('X', y)

  x_out = input

  # print(x_out)
  # print(y_out)
  # print(K)

  # for j in range(n):
  #   if j <= (K[0]+K[1])/2:
  #     x_out[j] = y_out[0]
  #   elif j > (K[obs-2]+K[obs-1])/2:
  #     x_out[j] = y_out[obs-1]
  #   else:
  #     for i in range(1,obs-1):
  #       if (K[i-1]+K[i])/2 < j <= (K[i]+K[i+1])/2:
  #         x_out[j] = y_out[i]
  
  for j in range(n):
    if j <= K[0]:
      x_out[j] = y_out[0]
    for k in range(obs-1):
      if j >= K[k] and j < K[k+1]:
        x_out[j] = y_out[k+1]
      if j >= K[k+1]:
        x_out[j] = y_out[obs-1]
      
  for i in range(obs):
    x_out[K[i]] = input_obs[i]

  return x_out


In [12]:
def rmse_cal(df, corr_mat, power):
  avail_ind = np.where(1-df.isna())
  avail_r_ind = avail_ind[0]
  avail_c_ind = avail_ind[1]
  l = len(avail_r_ind)
  masked_df = np.ma.masked_array(df, np.isnan(df))
  df_out = df.copy()
  for k in range(l):
    i = avail_r_ind[k]
    j = avail_c_ind[k]
    df_out.iloc[i,j] = np.ma.average(masked_df[i,:], weights=np.power(corr_mat[:,j],power))
  diff = df - df_out
  rmse = ((diff ** 2).sum().sum()/l) ** 0.5
  return rmse

In [13]:
def rmse_cal2(df, power):
  avail_ind = np.where(1-df.isna())
  avail_r_ind = avail_ind[0]
  avail_c_ind = avail_ind[1]
  l = len(avail_r_ind)

  masked_df = np.ma.masked_array(df, np.isnan(df))
  df_out = df.copy()

  w = np.array(df.corr())
  w[w<0]=0
  for i in range(len(w)):
    w[i,i]=0

  corr_mat = w

  for k in range(l):
    i = avail_r_ind[k]
    j = avail_c_ind[k]
    df_out.iloc[i,j] = np.ma.average(masked_df[i,:], weights=np.power(corr_mat[:,j],power))
  diff = df - df_out
  rmse = ((diff ** 2).sum().sum()/l) ** 0.5
  return rmse

In [14]:
def nested_quantile(df, a_min, a_max, n_level, RMSE_max):
  import numpy as np
  Q = np.zeros(5)
  Q[0] = a_min
  Q[4] = a_max
  beta = abs(Q[0]-Q[4])/4
  for j in range(1,4):
    Q[j] = Q[0] + j*beta

  # RMSE_L = rmse_calculator(df,Q[0])
  # RMSE_R = rmse_calculator(df,Q[4])
  RMSE_L = rmse_cal2(df,Q[0])
  RMSE_R = rmse_cal2(df,Q[4])

  # When RMSE_M=1, it is temporary
  RMSE_M = 1

  for i in range(n_level):

    RMSE = pd.DataFrame([[Q[0],RMSE_L],[Q[1],None],[Q[2],RMSE_M],[Q[3],None],[Q[4],RMSE_R]])

    for k in range(1,4):
      if k != 2 or RMSE_M == 1:
        # RMSE.iloc[k,1] =  rmse_calculator(df,Q[k])
        RMSE.iloc[k,1] =  rmse_cal2(df,Q[k])
      else:
        RMSE.iloc[k,1] = RMSE_M
    
    RMSE_sorted = RMSE.sort_values(by=[1])
        
    #print('RMSE Sorted:')
    #display(RMSE_sorted)
    
    alpha = RMSE_sorted.iloc[0,0]

    if RMSE_sorted.iloc[0,0] == Q[0]:
      Q[4] = Q[1]
      RMSE_R = RMSE.iloc[1,1] 
      RMSE_M = 1
    elif RMSE_sorted.iloc[0,0] == Q[4]:
      Q[0] = Q[3] 
      RMSE_L = RMSE.iloc[3,1]
      RMSE_M = 1
    elif RMSE_sorted.iloc[0,0] == Q[1]:
      Q[4] = Q[2]
      RMSE_M = RMSE.iloc[1,1]
      RMSE_R = RMSE.iloc[2,1]
    elif RMSE_sorted.iloc[0,0] == Q[2]:
      Q[0] = Q[1]
      Q[4] = Q[3]
      RMSE_L = RMSE.iloc[1,1]
      RMSE_M = RMSE.iloc[2,1]
      RMSE_R = RMSE.iloc[3,1]
    elif RMSE_sorted.iloc[0,0] == Q[3]:
      Q[0] = Q[2]
      RMSE_L = RMSE.iloc[2,1]
      RMSE_M = RMSE.iloc[3,1]

    beta = abs(Q[0]-Q[4])/4
    for j in range(1,4):
      Q[j] = Q[0] + j*beta

  return alpha

In [15]:
def power_finder(df, corr_mat, step_size, init_power, max_power):
  import math
  current_power = init_power
  opt_power = current_power
  opt_rmse = rmse_cal(df, corr_mat, opt_power)
  n_iter = math.floor((max_power-init_power)/step_size)
  for i in range(n_iter):
    current_power += step_size
    new_rmse = rmse_cal(df, corr_mat, current_power)
    if opt_rmse > new_rmse:
      opt_power = current_power
      opt_rmse = new_rmse
  return opt_power

In [16]:
#main loop 
import time as time_lib

k = 10 # k-fold validation 
n_items_list = [ 100, 100, 100, 100, 500, 500, 500, 500, 500, 1000, 1000, 1000, 1000, 1000]
m_users_list = [ 100, 300, 500, 700, 50, 100, 300, 500, 700,  50, 100, 300, 500, 700 ]
random_seed = 20230808

#prepare the result summary table 
result_summary_df = pd.DataFrame(columns = ['items','users','sparsity','avg_acc','std_acc','avg_rmse','std_rmse','avg_mad','std_mad','avg_time','std_time'])
# 6개의 열을 가진 빈 DataFrame 생성
result_detail_df = pd.DataFrame(columns=['n_items','m_users','acc', 'rmse', 'mad', 'time'])
#main 

for n, m in zip(n_items_list, m_users_list):
    n_items = n 
    n_users = m 

    ten = [str(j).zfill(2) for j in list(range(1,k+1))]
    
    labs = ['test'+i for i in ten]
    labs.insert(0,'orig')
    
    url_dict = {}
    url_dict["orig"] = './data/'+str(n_items)+'-by-'+str(n_users)+'_'+'original_matrix.csv'
    
    for i in ten:
        url_dict["test{0}".format(i)] = './data/'+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+i+'.csv'
    print(labs)
    
    for lab in labs:
        print(url_dict[lab])    
    
    df_dict = {}
    for lab in labs:
        df = pd.read_csv(url_dict[lab])
        df_dict[lab] = df.set_index('item_id')

    #compute the sparsity 
    n_row = df_dict["orig"].shape[0]
    n_col = df_dict["orig"].shape[1]
    Obs_ind = np.where(df_dict["orig"].notnull())    # Row and column indices for the observed entries of "Mdat"
    num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
    sparsity = 1 - num_Obs / (n_row * n_col)
    print("sparsity: "+str(sparsity))

    #Run algorithm 
    acc_list = []
    rmse_list = []
    mad_list = []
    time_list = []

    powers = [] #for monimpute

    df_orig = df_dict["orig"]
    df_orig.columns = range(df_orig.shape[1])

    corr_mat = df_orig.corr()
    temp_corr = np.copy(corr_mat)
    temp_corr[temp_corr < 0] = 0
    #print(np.round(temp_corr,2))
    #print(np.round(corr_mat,2))
    np.fill_diagonal(temp_corr, 0)
    orig_corr = np.copy(temp_corr)

    mm = df_orig.shape[0]
    nn = df_orig.shape[1]

    labs_test = labs[1:]

    a_min = 0
    a_max = 16
    n_level = 4
    RMSE_max = 1
    #col_max = [np.nanmax(df_dict["orig"].iloc[:, i].values) for i in range(df_dict["orig"].shape[1])]
    count = 0 

    for lab in labs_test:
        count += 1
        #print(lab)
        orig_corr = np.copy(temp_corr)
        df = df_dict[lab]
        
        masked_df = pd.DataFrame(np.ma.masked_array(df, np.isnan(df)))
        df_norm = normal1(masked_df)
        #df, maxs = normal2(masked_df)
        #df, maxs, probs = normal3(masked_df) #old version
        #df_norm = normal3(masked_df)

        start = time_lib.time()

        power = nested_quantile(df_norm, a_min, a_max, n_level, RMSE_max)
        powers.append(power)

        corr = np.power(orig_corr, power)  
        agg_mat = c_agg(df_norm, corr)

        df_agg_mat = pd.DataFrame(agg_mat)
        df_agg_mat.columns = ['agg'+str(i) for i in range(df.shape[1])]
        df_agg_mat.index = df.index
        df_agg = pd.concat([df, df_agg_mat], axis=1)
        #print(df_agg.shape)
        id_column = pd.DataFrame(range(mm))
        id_column.index = df_agg.index
        df_agg = pd.concat([id_column, df_agg], axis=1)
        #print(df_agg.shape)
        df_agg.columns = range(2*nn+1)
        
        

        for i in range(nn):
            df_agg_sorted = df_agg.sort_values(by=[df_agg.columns[nn+i+1],df_agg.columns[i+1]])
            index_i = df_agg_sorted.index
            vec = np.array(df_agg_sorted.iloc[:,i+1])
            out = monotonization(vec)
            out_list = []
            for i in range(len(out)):
                out_list.append(out[i])

            out_df = pd.DataFrame(out_list, index=index_i)
            df_agg = pd.concat([df_agg_sorted, out_df], axis=1)

        df_agg.columns = range(3*nn+1)
        df_agg_sorted_final = df_agg.sort_values(by=[df_agg.columns[0]])
        imputed = df_agg_sorted_final.iloc[:,2*nn+1:]

       
        end = time_lib.time()
        time = end - start

        df_orig.index = range(mm)
        imputed.index = range(mm)

        df_orig.columns = range(nn)
        imputed.columns = range(nn)

        #save the result data
        # url = './result/'

        # if count<10:
        #     filename = url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data0'+str(count)+'_mono_imputed.csv'
        # else:
        #     filename = url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data'+str(count)+'_mono_imputed.csv'
        
        # imputed.to_csv(filename)
        # print("saved the result file as "+str(filename))

        diff = df_orig - imputed
        sq_diff = diff ** 2
        abs_diff = abs(diff)

        n_match = 0
        for i in range(mm):
            for j in range(nn):
                if df_orig.isna().iloc[i,j]+df.isna().iloc[i,j]==1 and df_orig.iloc[i,j] == imputed.iloc[i,j]:
                    n_match += 1
        acc = n_match/(df.isna().sum().sum()-df_orig.isna().sum().sum())
        rmse = (sq_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())) ** 0.5
        mad = abs_diff.sum().sum()/(df.isna().sum().sum()-df_orig.isna().sum().sum())
        
        acc_list.append(acc)
        rmse_list.append(rmse)
        mad_list.append(mad)
        time_list.append(time)
        df = pd.DataFrame(data=[ acc_list, rmse_list,mad_list,time_list])
        #print(df)
        

    df = pd.DataFrame(data=[ acc_list, rmse_list,mad_list,time_list])
    display("n: "+str(n_items)+" ,m: "+ str(n_users))
    display(df.T)
    df_temp = df.T
    df_temp_2 = df_temp.copy()
    a = [n] * k 
    b = [m] * k
    df_temp_2.rename(columns={df_temp_2.columns[0]: 'acc', df_temp_2.columns[1]: 'rmse', df_temp_2.columns[2]: 'mad',df_temp_2.columns[3]: 'time'}, inplace=True)
    df_temp_2.insert(0,'m_users',b)
    df_temp_2.insert(0,'n_items',a) 

    # result_detail_df와 df_temp을 행 방향으로 이어 붙임
    result_detail_df = pd.concat([result_detail_df, df_temp_2], axis=0)

    #filename =url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'10_fold_test_data_all_missforest_imputed.csv'
    #df_temp.to_csv(filename)
    #print('saved '+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+ 'the summary result file as' +str(filename))
    # 열별 평균값 계산
    mean_values = df_temp.mean()
    # 열별 표준편차 계산
    std_values = df_temp.std()
    #new_row = [n,m,sparsity]+mean_values.tolist()
    new_row = [n,m,sparsity,mean_values[0],std_values[0],mean_values[1],std_values[1],mean_values[2],std_values[2],mean_values[3],std_values[3]]
    #display(new_row)
    result_summary_df = result_summary_df.append(pd.Series(new_row, index=result_summary_df.columns), ignore_index=True)
    #filename =url+str(random_seed)+'_'+str(n_items)+'-by-'+str(n_users)+'_'+'missforest_imputed.csv'
    #df_temp.to_csv(filename)
    #print("saved the summary result file as "+str(filename))
    display(result_summary_df)


display(result_summary_df)
display(result_detail_df)
        
#result_summary_df.to_csv('result_summary_df_mono.csv')
#result_detail_df.to_csv('result_detail_df_mono.csv')
        

['orig', 'test01', 'test02', 'test03', 'test04', 'test05', 'test06', 'test07', 'test08', 'test09', 'test10']
./data/100-by-100_original_matrix.csv
./data/20230808_100-by-100_10_fold_test_data01.csv
./data/20230808_100-by-100_10_fold_test_data02.csv
./data/20230808_100-by-100_10_fold_test_data03.csv
./data/20230808_100-by-100_10_fold_test_data04.csv
./data/20230808_100-by-100_10_fold_test_data05.csv
./data/20230808_100-by-100_10_fold_test_data06.csv
./data/20230808_100-by-100_10_fold_test_data07.csv
./data/20230808_100-by-100_10_fold_test_data08.csv
./data/20230808_100-by-100_10_fold_test_data09.csv
./data/20230808_100-by-100_10_fold_test_data10.csv
sparsity: 0.2914


'n: 100 ,m: 100'

,0,1,2,3
0,0.587571,0.765610,0.464689,15.298200
1,0.564972,0.808675,0.501412,17.149118
2,0.507062,0.857832,0.566384,14.655539
3,0.535311,0.842883,0.538136,14.563936
4,0.547250,0.803729,0.510578,14.857239
5,0.520451,0.884756,0.574048,14.719637
6,0.551481,0.810718,0.513399,16.880854
7,0.534556,0.843125,0.541608,16.470784
8,0.547250,0.840612,0.531735,15.785620
9,0.588152,0.795793,0.480959,16.791864


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,100.0,100.0,0.2914,0.548405,0.026361,0.825373,0.034709,0.522295,0.034984,15.717279,1.027366


['orig', 'test01', 'test02', 'test03', 'test04', 'test05', 'test06', 'test07', 'test08', 'test09', 'test10']
./data/100-by-300_original_matrix.csv
./data/20230808_100-by-300_10_fold_test_data01.csv
./data/20230808_100-by-300_10_fold_test_data02.csv
./data/20230808_100-by-300_10_fold_test_data03.csv
./data/20230808_100-by-300_10_fold_test_data04.csv
./data/20230808_100-by-300_10_fold_test_data05.csv
./data/20230808_100-by-300_10_fold_test_data06.csv
./data/20230808_100-by-300_10_fold_test_data07.csv
./data/20230808_100-by-300_10_fold_test_data08.csv
./data/20230808_100-by-300_10_fold_test_data09.csv
./data/20230808_100-by-300_10_fold_test_data10.csv
sparsity: 0.4202


...

,0,1,2,3
0,0.482517,0.972710,0.637807,282.880629
1,0.476634,0.969395,0.643135,276.457601
2,0.484515,0.973509,0.638473,276.095120
3,0.489399,0.954859,0.626485,275.492229
4,0.479520,0.980439,0.646465,275.926764
5,0.481740,0.973452,0.640138,274.213229
6,0.484182,0.970825,0.638806,280.343514
7,0.489345,0.967909,0.631853,276.698179
8,0.486903,0.961120,0.631410,276.494587
9,0.490788,0.956374,0.625416,275.374396


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,100.0,100.0,0.291400,0.548405,0.026361,0.825373,0.034709,0.522295,0.034984,15.717279,1.027366
1,100.0,300.0,0.420200,0.579452,0.015694,0.757773,0.022711,0.468209,0.020723,45.271554,3.815046
2,100.0,500.0,0.526960,0.528708,0.006488,0.944423,0.016733,0.587858,0.011422,62.702900,4.799226
3,100.0,700.0,0.609329,0.454053,0.005665,1.162626,0.015593,0.764727,0.009151,79.469158,2.839325
4,500.0,50.0,0.466640,0.447056,0.019977,0.981047,0.045164,0.676460,0.038881,30.822203,2.597719
5,500.0,100.0,0.530200,0.479438,0.010070,0.910448,0.017923,0.615155,0.014777,49.134668,1.080049
6,500.0,300.0,0.661573,0.509258,0.005864,0.865888,0.011595,0.570858,0.009403,122.621723,1.987678
7,500.0,500.0,0.740172,0.521191,0.005691,0.862047,0.010789,0.557877,0.008405,180.556329,3.371153
8,500.0,700.0,0.792409,0.490331,0.007684,0.966465,0.012623,0.630001,0.012203,232.225962,4.049007
9,1000.0,50.0,0.633380,0.436692,0.006630,1.021857,0.025752,0.705799,0.018537,50.228419,1.435016


,items,users,sparsity,avg_acc,std_acc,avg_rmse,std_rmse,avg_mad,std_mad,avg_time,std_time
0,100.0,100.0,0.291400,0.548405,0.026361,0.825373,0.034709,0.522295,0.034984,15.717279,1.027366
1,100.0,300.0,0.420200,0.579452,0.015694,0.757773,0.022711,0.468209,0.020723,45.271554,3.815046
2,100.0,500.0,0.526960,0.528708,0.006488,0.944423,0.016733,0.587858,0.011422,62.702900,4.799226
3,100.0,700.0,0.609329,0.454053,0.005665,1.162626,0.015593,0.764727,0.009151,79.469158,2.839325
4,500.0,50.0,0.466640,0.447056,0.019977,0.981047,0.045164,0.676460,0.038881,30.822203,2.597719
5,500.0,100.0,0.530200,0.479438,0.010070,0.910448,0.017923,0.615155,0.014777,49.134668,1.080049
6,500.0,300.0,0.661573,0.509258,0.005864,0.865888,0.011595,0.570858,0.009403,122.621723,1.987678
7,500.0,500.0,0.740172,0.521191,0.005691,0.862047,0.010789,0.557877,0.008405,180.556329,3.371153
8,500.0,700.0,0.792409,0.490331,0.007684,0.966465,0.012623,0.630001,0.012203,232.225962,4.049007
9,1000.0,50.0,0.633380,0.436692,0.006630,1.021857,0.025752,0.705799,0.018537,50.228419,1.435016


,n_items,m_users,acc,rmse,mad,time
0,100,100,0.587571,0.765610,0.464689,15.298200
1,100,100,0.564972,0.808675,0.501412,17.149118
2,100,100,0.507062,0.857832,0.566384,14.655539
3,100,100,0.535311,0.842883,0.538136,14.563936
4,100,100,0.547250,0.803729,0.510578,14.857239
...,...,...,...,...,...,...
5,1000,700,0.481740,0.973452,0.640138,274.213229
6,1000,700,0.484182,0.970825,0.638806,280.343514
7,1000,700,0.489345,0.967909,0.631853,276.698179
8,1000,700,0.486903,0.961120,0.631410,276.494587


: 